# Populate Workspace Data

The suite of notebooks in this directory demonstrates how to perform Geostatistical modeling operations using Seequent Evo's compute task functionality. Prior to executing any compute tasks it is necessary to populate the Evo workspace with data. 

## Requirements

You must have a Seequent account with the Evo entitlement. You'll need:
- The client ID of your Evo application
- The callback/redirect URL of your Evo application

To obtain these credentials, refer to the [Apps and tokens guide](https://developer.seequent.com/docs/guides/getting-started/apps-and-tokens).

---

First, import the required Python packages, including the [evo-python-sdk](https://github.com/SeequentEvo/evo-python-sdk), which facilitates authentication, communication, and data exchange between your local computer and Seequent Evo.

In [ ]:
import pandas as pd
from dotenv import dotenv_values
from helpers import variogram_structure

import evo.objects.typed as objs
from evo.blockmodels.typed import Units
from evo.notebooks import ServiceManagerWidget
from evo.widgets import get_viewer_url_for_objects

# Load the widgets extension for rich HTML display of Evo objects
%load_ext evo.widgets

Authentication is performed using the `ServiceManagerWidget`, which returns a Python object that contains everything needed to connect to Evo services.

Create a local `.env` file alongside this notebook using `.env.example` as a template, and set `client_id` and `redirect_url` for your application. The redirect URL must match one registered for the application. Local `.env` files are Git-ignored.

`dotenv_values` reads these settings into a dictionary without modifying environment variables. Its keys are passed directly to `with_auth_code`, so they must match the method's parameter names. Keep application-specific values in `.env`, not in notebook cells or outputs.

In [ ]:
evo_connection = await ServiceManagerWidget.with_auth_code(**dotenv_values()).login()

## First Upload the Drillholes

This example assumes the data has already been composited and that the XYZ coordinates represent the midpoint of each composite interval. Load the data from CSV into a [pandas DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) and inspect the first few rows.


In [ ]:
df = pd.read_csv("sample-data/WP_assay_with_GM.csv", dtype={"Geological_model": "category"})
df.head()

A quick statistical summary can be helpful, especially with unfamiliar data.

In [ ]:
df.describe()

## Upload Composites to Seequent Evo

In this case, a [pointset object](https://developer.seequent.com/docs/data-structures/geoscience-objects/schemas/objects/pointset) is appropriate because the data has midpoint locations without from/to information. If provided with from/to information, consider a [downhole intervals object](https://developer.seequent.com/docs/data-structures/geoscience-objects/schemas/objects/downhole-intervals).

To generate the pointset object, first create the `PointSetData` object, then upload it to create a `PointSet` object on Seequent Evo. Once created, the pointset can be accessed on the Seequent Evo portal via the `Portal` and `Viewer` links.

In [ ]:
# create the data object locally
pointset_data = objs.PointSetData(
    name="WP comps",
    description="Composited Cu and Au assays with Density",
    locations=df,
    coordinate_reference_system="EPSG:32650",
)

# upload and create the pointset within Seequent Evo workspace
pointset = await objs.PointSet.create(evo_connection, pointset_data)
pointset

## Upload a Variogram Model

Variograms can be created in Leapfrog Edge and published to Evo or created using known parameters.

In this case, the variogram model is constructed from two spherical structures that share the same rotation (`is_rotation_fixed=True`). This model is generated in the original units (not standardized or normal-score transformed), so both the `sill` and the `data_variance` are equal.

To simplify structure creation, the `variogram_structure` function from `helpers.py` selects the structure type by name.

In [ ]:
structures = [
    variogram_structure("spherical", contribution=0.25, major=80.0, semi_major=60.0, minor=40.0),
    variogram_structure("spherical", contribution=0.51, major=250.0, semi_major=180.0, minor=100.0),
]

Next, compile the structures along with other pertinent data and metadata into the `VariogramData` object for upload to the Seequent Evo workspace.

In [ ]:
variogram_data = objs.VariogramData(
    name="Cu_pct Variogram Model",
    description="Variogram model for Cu_pct in original units",
    sill=0.84,
    nugget=0.08,
    is_rotation_fixed=True,
    modelling_space="data",
    data_variance=0.84,
    attribute="Cu_pct",
    structures=structures,
)

# Now, create the variogram object in the Evo workspace
variogram = await objs.Variogram.create(evo_connection, variogram_data)
variogram

## Create a Block Model

A regular block model will be the target of these geostatistical tasks. Similar to the examples above, first create the `RegularBlockModelData` object, then upload it to Evo to generate the `BlockModel` object.

In [ ]:
df = pd.read_parquet("sample-data/WP Regular BlockModel.parquet")
df.head()

In [ ]:
blockmodel_data = objs.RegularBlockModelData(
    name="WP Regular BlockModel",
    description="Regular block model for the WP deposit",
    origin=(444700, 492800, 2330),
    n_blocks=(110, 160, 96),
    block_size=(10.0, 10.0, 10.0),
    size_unit_id=Units.METRES,
    cell_data=df,
    coordinate_reference_system="EPSG:32650",
)

block_model = await objs.BlockModel.create_regular(evo_connection, blockmodel_data)
block_model

Finally, visualize and inspect the objects using the Evo viewer.

In [ ]:
get_viewer_url_for_objects(evo_connection, [pointset, variogram, block_model])

Now that the Evo workspace has been populated with the necessary input composites and variogram, along with a block model to store results, continue with the geostatistical modeling examples in this directory.